# Assemble datasets from simulations
Combine data from simulations of different network architectures

In [1]:
import numpy as np
import pandas as pd
import os
import pickle
from tqdm import tqdm
from joblib import Parallel, delayed

from stoch_sim_model import *

In [2]:
# Set parameters
sim_kind = 'agent'
infection_type = 'prim'
reg_model = ''
runs = '-1-'
comment = "full-reg-vir"

d = '/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/'

sim_sum_list = []
parameters_nets = []
prim_diff_bias_list = []
#sec_diff_bias_list = []
cell_series_list = []
# lineage_diff_nets = []

In [4]:
# Figure out which jobs didn't run:
d_rerun = '/gscratch/scrubbed/oukogu/slurm_output/'
run_list = [int(f[6:][:-4]) for f in os.listdir(d_rerun) if 'batch' in f]
out = [str(x) for x in [k for k in np.arange(0, 1000)] if x not in run_list]
(' '.join((out)))

''

In [5]:
# # Load data from second infections
# file_list = [f for f in os.listdir(os.path.join(d, "raw"))  if runs in f and infection_type in f and comment in f]

# for f in tqdm(file_list):
#     filepath = os.path.join(os.path.join(d, "raw"), f)
#     with open(filepath, 'rb') as filename:  
#         import_dict = pickle.load(filename)

#     parameters_nets = []
#     prim_diff_bias_list = []
#     #sec_diff_bias_list = []
#     cell_series_list = []
#     # mean_prim_diff_bias = []
#     # std_prim_diff_bias = []
#     # mean_sec_diff_bias = []
#     # std_sec_diff_bias = []
#     # mean_cell_series = []
#     # std_cell_series = []
#     # mean_lineage_diff = []
#     # std_lineage_diff = []

#     parameters = np.array(import_dict["parameters"])
#     # prim_diff_bias = np.array(import_dict["prim_diff_bias"])
#     # sec_diff_bias = np.array(import_dict["sec_diff_bias"])
#     # cell_series = np.array(import_dict["cell_time_series"])
#     # lineage_diff = np.array(import_dict["lineage_diff"])
#     sim_sum = np.array(import_dict["summary_stats"])

#     # virs = np.unique(parameters[:,[4,7,13,14]], axis = 0)
    
#     # for i, vir in enumerate(virs): # Need to fix this to stop averaging over K_EI and K_EH
#     #     index = (parameters[:,4] == vir[0])*(parameters[:,7] == vir[1])*(parameters[:,13] == vir[2])*(parameters[:,14] == vir[3])
                
#     sim_sum_list.append(np.hstack((sim_sum, parameters)))

#         # mean_prim_diff_bias.append(np.mean(prim_diff_bias[index], axis = 0))
#         # std_prim_diff_bias.append(np.std(prim_diff_bias[index], axis = 0))

#         # mean_sec_diff_bias.append(np.mean(sec_diff_bias[index], axis = 0))
#         # std_sec_diff_bias.append(np.std(sec_diff_bias[index], axis = 0))
        
#         # mean_cell_series.append(np.mean(cell_series[index], axis = 0))
#         # std_cell_series.append(np.std(cell_series[index], axis = 0))
        
#         # mean_lineage_diff.append(np.mean(lineage_diff[index], axis = 0))
#         # std_lineage_diff.append(np.std(lineage_diff[index], axis = 0))

# # Save datasets
# ### (1) Summary stats
# np.save(os.path.join(d, "raw",comment+"_summary_stats"), np.vstack(sim_sum_list))
#     # np.save(os.path.join(d, "summary_stats","std",f[:-4]), std_sim_sum)
#     # ### (2) Differentiation bias
#     # np.save(os.path.join(d, "prim_diff_bias","mean",f[:-4]), mean_prim_diff_bias)
#     # np.save(os.path.join(d, "prim_diff_bias","std",f[:-4]), std_prim_diff_bias)
#     # np.save(os.path.join(d, "sec_diff_bias","mean",f[:-4]), mean_sec_diff_bias)
#     # np.save(os.path.join(d, "sec_diff_bias","std",f[:-4]), std_sec_diff_bias)
#     # ### (3) Cell time series
#     # np.save(os.path.join(d, "cell_time_series","mean",f[:-4]), mean_cell_series)
#     # np.save(os.path.join(d, "cell_time_series","std",f[:-4]), std_cell_series)
#     # ### (4) Lineage differentiation
#     # np.save(os.path.join(d, "lineage_diff","mean",f[:-4]), mean_lineage_diff)
#     # np.save(os.path.join(d, "lineage_diff","std",f[:-4]), std_lineage_diff)

In [5]:
num_cpu = 20
num_files = 1000
file_list = [f for f in os.listdir(os.path.join(d, "raw"))  if runs in f and infection_type in f and comment in f]

def import_dict_func(f,d):
    
    file_path = os.path.join(os.path.join(d, "raw"), f)
    with open(file_path, 'rb') as filename:  
        import_dict = pickle.load(filename)

    parameters = np.array(import_dict["parameters"])
    # p_mem_survived = np.array(import_dict["pmemory_survived"])
    sim_sum = np.array(import_dict["summary_stats"])
    # sim_sum[:,14] = p_mem_survived
    

    out = np.hstack((parameters, sim_sum))

    return out

full_summary_data = Parallel(n_jobs = num_cpu, batch_size = max(int(num_files/num_cpu),1))(delayed(import_dict_func)(f = file_name, d = d)
                                                    for file_name in file_list)

/opt/minimamba/envs/maximmune/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


In [6]:
for i, a in enumerate(full_summary_data):
    if a.shape[0] < 60:
        print(i)

In [7]:
# Save datasets
### (1) Summary stats
np.save(os.path.join(d, "raw", "stacked_data"+runs+"runs"+'-'+comment), np.vstack(full_summary_data))

In [4]:
# # Save stacked datasets
# d_mean = '/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/summary_stats/mean/'
# d_std = '/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/summary_stats/std/'

# mean_nets = []
# std_nets = []

# file_list = [f for f in os.listdir(d_mean) if runs in f and infection_type in f and comment in f]

# for f in tqdm(file_list):
#     net_mean = np.load(os.path.join(d_mean, f))
#     net_std = np.load(os.path.join(d_std, f))
#     mean_nets.append(net_mean)
#     std_nets.append(net_std)

# mean_data = np.vstack(mean_nets)
# std_data = np.vstack(std_nets)

# np.save(os.path.join(d_mean, "stacked_data"+runs+"runs"+'-'+comment), mean_data)
# np.save(os.path.join(d_std, "stacked_data"+runs+"runs"+'-'+comment), std_data)

  1%|██                                                                                                                                              | 666/47543 [00:44<52:05, 15.00it/s]


ValueError: cannot reshape array of size 0 into shape (5,61)

In [5]:
f

'0.4-1.6--0.0--0.8-0.40--0.7-10-sec-NM-reg-EM-reg.npy'